In [1]:
!nvidia-smi

import psutil
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

Tue Sep  1 16:49:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

# 1. Check if the Kaggle GPU is active
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(def_device := f"Using hardware device: {device}")

# 2. When loading your Hugging Face embedding or LLM model, pass it to the device:
# Example:
# model = AutoModelForCausalLM.from_pretrained("your-model").to(device)


Using hardware device: cuda


In [3]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System RAM: {ram_gb:.1f} GB")

System RAM: 31.3 GB


In [4]:
!pip -q install -U datasets pandas pyarrow tqdm sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 105.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 39.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 44.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but 

In [5]:
from huggingface_hub import login

login()


In [6]:
from datasets import load_dataset

ds = load_dataset(
    "overthelex/indian-court-decisions",
    "high_courts",
    split="train",
    streaming=True
)

print(ds)

README.md: 0.00B [00:00, ?B/s]

IterableDataset({
    features: ['cnr', 'source', 'court_code', 'court_name', 'bench', 'year', 'full_text', 'text_length', 'title', 'judge', 'petitioner', 'respondent', 'decision_date', 'disposal_nature', 'disposal_nature_normalized', 'case_type'],
    num_shards: 1
})


In [7]:
from datasets import load_dataset

ds = load_dataset(
    "overthelex/indian-court-decisions",
    "high_courts",
    split="train",
    streaming=True
)

# Shuffle only the stream buffer, not the whole dataset
ds = ds.shuffle(seed=42, buffer_size=10_000)

print(ds)

IterableDataset({
    features: ['cnr', 'source', 'court_code', 'court_name', 'bench', 'year', 'full_text', 'text_length', 'title', 'judge', 'petitioner', 'respondent', 'decision_date', 'disposal_nature', 'disposal_nature_normalized', 'case_type'],
    num_shards: 1
})


In [8]:
for i, row in enumerate(ds):
    print(row.keys())
    print(row["cnr"])
    print(row["decision_date"])
    print(len(row["full_text"]))
    break

dict_keys(['cnr', 'source', 'court_code', 'court_name', 'bench', 'year', 'full_text', 'text_length', 'title', 'judge', 'petitioner', 'respondent', 'decision_date', 'disposal_nature', 'disposal_nature_normalized', 'case_type'])
BRHC011186892019_1_2023-02-01
2023-02-01
5801


In [9]:
from datasets import load_dataset
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

# -----------------------------
# Configuration
# -----------------------------
TARGET = 50_000
SHARD_SIZE = 5_000
OUTPUT_DIR = Path("/content/legalrag_corpus")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Stream dataset
# -----------------------------
ds = load_dataset(
    "overthelex/indian-court-decisions",
    "high_courts",
    split="train",
    streaming=True
)

ds = ds.shuffle(seed=42, buffer_size=10_000)

# -----------------------------
# Collection
# -----------------------------
seen_texts = set()
buffer = []
total_saved = 0
shard_id = 0

with tqdm(total=TARGET, desc="Collecting judgments") as pbar:

    for row in ds:

        if total_saved >= TARGET:
            break

        text = row.get("full_text")

        if not text:
            continue

        text = str(text).strip()

        # Exact duplicate removal
        if text in seen_texts:
            continue

        seen_texts.add(text)

        record = {
            "cnr": row.get("cnr"),
            "source": row.get("source"),
            "court_code": row.get("court_code"),
            "court_name": row.get("court_name"),
            "bench": row.get("bench"),
            "year": row.get("year"),
            "full_text": text,
            "text_length": row.get("text_length"),
            "title": row.get("title"),
            "judge": row.get("judge"),
            "petitioner": row.get("petitioner"),
            "respondent": row.get("respondent"),
            "decision_date": row.get("decision_date"),
            "disposal_nature": row.get("disposal_nature"),
            "disposal_nature_normalized": row.get(
                "disposal_nature_normalized"
            ),
            "case_type": row.get("case_type"),
        }

        buffer.append(record)
        total_saved += 1
        pbar.update(1)

        # Write shard
        if len(buffer) >= SHARD_SIZE:

            df = pd.DataFrame(buffer)

            path = OUTPUT_DIR / f"judgments_{shard_id:03d}.parquet"

            df.to_parquet(path, index=False)

            print(f"\nSaved {path}")

            buffer.clear()
            shard_id += 1

# Write final partial shard
if buffer:
    df = pd.DataFrame(buffer)

    path = OUTPUT_DIR / f"judgments_{shard_id:03d}.parquet"

    df.to_parquet(path, index=False)

    print(f"\nSaved {path}")

print("\nCollection complete")
print("Total unique judgments:", total_saved)


Saved /content/legalrag_corpus/judgments_000.parquet

Saved /content/legalrag_corpus/judgments_001.parquet

Saved /content/legalrag_corpus/judgments_002.parquet

Saved /content/legalrag_corpus/judgments_003.parquet

Saved /content/legalrag_corpus/judgments_004.parquet

Saved /content/legalrag_corpus/judgments_005.parquet

Saved /content/legalrag_corpus/judgments_006.parquet

Saved /content/legalrag_corpus/judgments_007.parquet

Saved /content/legalrag_corpus/judgments_008.parquet

Saved /content/legalrag_corpus/judgments_009.parquet

Collection complete
Total unique judgments: 50000


In [10]:
import pandas as pd
from pathlib import Path

files = sorted(Path("/content/legalrag_corpus").glob("*.parquet"))

df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print("Rows:", len(df))
print("Unique texts:", df["full_text"].nunique())
print("\nCourts:")
print(df["court_code"].value_counts())

print("\nYears:")
print(df["decision_date"].astype(str).str[:4].value_counts().sort_index())

Rows: 50000
Unique texts: 50000

Courts:
court_code
18_6     12491
10_8     11877
24_17     8973
19_16     5661
16_20     3307
14_25     2376
17_21     1859
27_1       905
36_29      485
22_18      454
3_22       397
11_24      322
23_23      305
1_12       180
5_15       151
8_9         52
21_11       49
33_10       47
7_26        46
2_5         33
20_7        10
32_4         9
29_3         7
28_2         2
9_13         2
Name: count, dtype: int64

Years:
decision_date
1996     2791
1998     3292
1999     2637
2000      271
2001      923
2002       72
2008      598
2009       26
2012      231
2013        4
2015     8425
2016       41
2017        8
2019     1295
2020     4503
2021       93
2022       12
2023     8716
2024      276
2025    15785
2026        1
Name: count, dtype: int64


In [11]:
print(df[["court_code", "court_name", "source"]]
      .drop_duplicates()
      .sort_values("court_code")
      .to_string(index=False))

court_code court_name source
      10_8                hc
     11_24                hc
     14_25                hc
     16_20                hc
     17_21                hc
      18_6                hc
     19_16                hc
      1_12                hc
      20_7                hc
     21_11                hc
     22_18                hc
     23_23                hc
     24_17                hc
      27_1                hc
      28_2                hc
      29_3                hc
       2_5                hc
      32_4                hc
     33_10                hc
     36_29                hc
      3_22                hc
      5_15                hc
      7_26                hc
       8_9                hc
      9_13                hc


In [12]:
print(
    df.groupby("court_code")["text_length"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("count", ascending=False)
      .to_string()
)

            count          mean  median   min     max
court_code                                           
18_6        12491   5901.048995  2807.0  1001  296380
10_8        11877   3686.813253  2879.0  1001  951318
24_17        8973   7117.465953  4523.0  1001  254068
19_16        5661   3365.923865  2238.0  1001  217187
16_20        3307   9613.992743  5012.0  1003  223399
14_25        2376   7811.414562  2723.0  1001  210884
17_21        1859   7166.510490  3111.0  1002  220388
27_1          905   4866.230939  2111.0  1001  121006
36_29         485   5207.995876  3356.0  1002   84330
22_18         454   4348.110132  3048.5  1013   73710
3_22          397   5925.549118  3206.0  1001  104156
11_24         322  15732.170807  8428.5  1007  170609
23_23         305   4442.740984  3111.0  1009   63064
1_12          180  12195.094444  7712.5  1017  145375
5_15          151   2120.298013  1484.0  1006   18533
8_9            52   6766.307692  3332.5  1000   39335
21_11          49   6063.632

In [13]:
import re

def extract_court(text):
    text = str(text)[:1500]

    patterns = [
        r"IN THE (?:HIGH COURT|HIGH COURT OF) ([A-Z][A-Z ]+)",
        r"IN THE HIGH COURT OF ([A-Z][A-Z ]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1).strip()

    return "UNKNOWN"


court_map = (
    df.assign(extracted_court=df["full_text"].apply(extract_court))
      .groupby("court_code")["extracted_court"]
      .agg(lambda x: x.value_counts().index[0])
      .sort_index()
)

print(court_map.to_string())

court_code
10_8       OF JUDICATURE AT PATNA CRIMINAL MISCELLANEOUS N
11_24                                              UNKNOWN
14_25                              OF MANIPUR AT IMPHAL WP
16_20                                              UNKNOWN
17_21                                              UNKNOWN
18_6                                               UNKNOWN
19_16                                              UNKNOWN
1_12                                               UNKNOWN
20_7                                               UNKNOWN
21_11                                              UNKNOWN
22_18                                              UNKNOWN
23_23                                              UNKNOWN
24_17    OF GUJARAT AT AHMEDABAD SPECIAL CIVIL APPLICAT...
27_1           OF JUDICATURE AT BOMBAY BENCH AT AURANGABAD
28_2                                               UNKNOWN
29_3                                               UNKNOWN
2_5                                    OF HIM

In [14]:
df.head()

,cnr,source,court_code,court_name,bench,year,full_text,text_length,title,judge,petitioner,respondent,decision_date,disposal_nature,disposal_nature_normalized,case_type
0,BRHC011186892019_1_2023-02-01,hc,10_8,,patnahcucisdb94,2023,IN THE HIGH COURT OF JUDICATURE AT PATNA Civil...,5801,CWJC/226/2020 of Shashank Shekhar @ Shashank V...,MR. JUSTICE SANJEEV PRAKASH SHARMA,,,2023-02-01,DISMISSED,dismissed,
1,BRHC011266752025_1_2025-12-18,hc,10_8,,patnahcucisdb94,2025,IN THE HIGH COURT OF JUDICATURE AT PATNA CRIMI...,3048,CR. MISC./87109/2025 of Sunil Kumar @ Sunil Pa...,MR. JUSTICE ASHOK KUMAR PANDEY,,,2025-12-18,ALLOWED,allowed,
2,MLHC010005242017_1_2020-03-11,hc,17_21,,meghalaya,2020,WP(C) No. 198 of 2017 with WP(C) No. 257 of 20...,10720,WP(C)/198/2017 of Hillford Thangkhiew Vs State...,HON'BLE MR. JUSTICE H. S. THANGKHIEW,,,2020-03-11,Disposed Off,disposed,WP
3,HCBM030297452007_1_2019-07-13,hc,27_1,,hcaurdb,2019,( 1 ) 927 FA 3654-08.odt IN THE NATIONAL LOK A...,1680,FA/3651/2008 of GODAWARI MARATHWADA IRRIGATION...,SHRI JUSTICE J P DEVADHAR,,,2019-07-13,Withdrawn,withdrawn,FA
4,GAHC030001292023_1_2025-03-27,hc,18_6,,azghccis,2023,Page No.# 1/9 GAHC030004172021 THE GAUHATI HIG...,4601,I.A.(Civil)/29/2023 of B.Chhasa and 23 Ors. Vs...,HONOURABLE MRS. JUSTICE MARLI VANKUNG,,,2023-03-22,Allowed,allowed,


In [15]:
import re
import pandas as pd

def quality_stats(text):
    text = str(text)

    bad_chars = len(re.findall(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', text))
    replacement_chars = text.count("�")

    return pd.Series({
        "chars": len(text),
        "control_chars": bad_chars,
        "replacement_chars": replacement_chars,
        "newlines": text.count("\n"),
    })


sample_stats = df["full_text"].sample(
    min(5000, len(df)),
    random_state=42
).apply(quality_stats)

print(sample_stats.describe().to_string())

               chars  control_chars  replacement_chars  newlines
count    5000.000000    5000.000000             5000.0    5000.0
mean     5328.916200       9.508800                0.0       0.0
std      8050.413014     175.527792                0.0       0.0
min      1001.000000       0.000000                0.0       0.0
25%      1957.750000       0.000000                0.0       0.0
50%      2912.500000       0.000000                0.0       0.0
75%      5073.000000       0.000000                0.0       0.0
max    220388.000000    7478.000000                0.0       0.0


# preprocessing

In [16]:
# identify the bad documents
bad = df[df["full_text"].apply(
    lambda x: len(re.findall(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', str(x)))
) > 0].copy()

bad["control_chars"] = bad["full_text"].apply(
    lambda x: len(re.findall(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', str(x)))
)

print("Documents with control characters:", len(bad))
print("\nTop 20:")
print(
    bad[["cnr", "court_code", "decision_date", "text_length", "control_chars"]]
    .sort_values("control_chars", ascending=False)
    .head(20)
    .to_string(index=False)
)

Documents with control characters: 474

Top 20:
                          cnr court_code decision_date  text_length  control_chars
GAHC040002732002_1_2002-08-27       18_6    2002-08-27        50933           9758
GAHC040008452001_1_2002-09-11       18_6    2002-09-11        42048           8825
GAHC040009462014_1_2015-01-09       18_6    2015-01-09        43858           8779
GAHC040009962014_1_2015-09-10       18_6    2015-09-10        39402           8178
GAHC040001942012_1_2015-09-19       18_6    2015-09-19        34230           7478
GAHC040004942014_1_2015-10-16       18_6    2015-10-16        30548           6371
JKHC020023982013_1_2015-02-09       1_12    2015-02-09         9652           6116
GAHC040009332014_1_2015-03-18       18_6    2015-03-18        28356           6040
GAHC040005642014_1_2015-03-18       18_6    2015-03-18        28356           6040
GAHC040004552013_1_2015-03-18       18_6    2015-03-18        29862           6006
GAHC040008172015_1_2015-08-24       18_

In [17]:
# worst one
row = bad.sort_values("control_chars", ascending=False).iloc[0]

print("CNR:", row["cnr"])
print("Control chars:", row["control_chars"])
print("\nFirst 3000 characters:\n")
print(repr(row["full_text"][:3000]))

CNR: GAHC040002732002_1_2002-08-27
Control chars: 9758

First 3000 characters:

":3 & \x03\x16\x15\x1a\x12\x15\x13\x13\x15 %()25( 7+(\x03+21 %/(\x0305\x11\x03-867,&(\x03,\x11\x03$\x11\x03$16$5, \x03\x14\x03 \x03&ODLPLQJ\x03WR\x03EH\x03SHUSOH[HG\x03DW\x03WKH\x03GHFLVLRQ\x03RI\x03WKH\x036WDWH\x03*RYHUQPHQW\x03DQG\x03FKDOO HQJLQJ\x03WKH\x03YLUXV\x03DQG\x03FRQVWLWXWLRQDOLW\\\x03RI\x03WKH\x036WDWH\x03FDELQHW V\x03GHFLVLRQ\x03WR\x03VHOHFW DQG\x03QRPLQDWH\x03FDQGLGDWHV\x03IRU\x03SXUVXLQJ\x03WKH\x03FRXUVHV\x03RI\x03PEEV\x03DQG\x03%'6\x03RQ\x03WKH\x03EDVLV\x03R I\x03D\x03SROLF\\\x0f\x03ZKLFK\x03SURYLGHV\x03IRU\x03PDNLQJ\x03VHOHFWLRQ\x03DQG\x03QRPLQDWLRQ\x03RI\x03WKRVH\x03FDQGLGDW HV\x0f\x03ZKR\x03GR\x03QRW\x03FRPH\x03ZLWKLQ\x03WKH\x03]RQH\x03RI\x03FRQVLGHUDWLRQ\x03RQ\x03WKH\x03EDVLV\x03RI\x03WKH\x03PHULW\x03O LVW\x03SUHSDUHG\x03IROORZLQJ\x03WKH\x03-RLQW\x03(QWUDQFH\x03([DPLQDWLRQ\x03 KHUHLQDIWHU\x0f\x03LQ\x03VKRUW\x0f\x03FD OOHG\x03DV\x03?WKH\x03-((? \x0f\x03EXW\x03DUH\x03IURP\x03VXFK\x03GLVWUL

In [18]:
text = df[df["cnr"] == "GAHC040002732002_1_2002-08-27"]["full_text"].iloc[0]

def caesar_decode(text, shift=3):
    result = []

    for ch in text:
        if 'A' <= ch <= 'Z':
            result.append(chr((ord(ch) - ord('A') - shift) % 26 + ord('A')))
        elif 'a' <= ch <= 'z':
            result.append(chr((ord(ch) - ord('a') - shift) % 26 + ord('a')))
        else:
            result.append(ch)

    return ''.join(result)

decoded = caesar_decode(text)

print(decoded[:3000])

:3 &  %()25( 7+(+21 %/(05-867,&(,$$16$5,  &LAIMINGTOBEPERPLE[EDATTHEDECISIONOFTHE6TATE*OVERNMENTANDCHALL ENGINGTHEVIRUSANDCONSTITUTIONALIT\OFTHE6TATECABINET SDECISIONTOSELECT ANDNOMINATECANDIDATESFORPURSUINGTHECOURSESOFMBBSAND%'6ONTHEBASISO FAPOLIC\WHICHPROVIDESFORMAKINGSELECTIONANDNOMINATIONOFTHOSECANDIDAT ESWHODONOTCOMEWITHINTHE]ONEOFCONSIDERATIONONTHEBASISOFTHEMERITL ISTPREPAREDFOLLOWINGTHE-OINT(NTRANCE([AMINATION HEREINAFTERINSHORTCA LLEDAS?THE-((? BUTAREFROMSUCHDISTRICTS S FROMWHICHNOCANDIDATECOUL DBESELECTEDONTHEBASISOFMERITALONEFORPURSUINGTHESAIDCOURSESTHEPRI NCIPLETHUSBEINGTHATEACHDISTRICTOFTHE6TATEMUSTGETREPRESENTATIONIFA N\OFITSCANDIDATEHASRECEIVEDTHEREQUISITEQUALIF\INGMARKSTHOUGHHEMA\B EFARLOWERTHANOTHERSINTHEMERITLISTTHEPETITIONERSHAVEAPPROACHEDTHIS &OURT  %\THISAPPLICATIONMADEUNDERARTICLE

In [19]:
# how much of the corrupted text can be recovered.
import re

text = df[df["cnr"] == "GAHC040002732002_1_2002-08-27"]["full_text"].iloc[0]

def caesar_letters(text, shift=-3):
    result = []

    for ch in text:
        if 'A' <= ch <= 'Z':
            result.append(chr((ord(ch) - ord('A') + shift) % 26 + ord('A')))
        elif 'a' <= ch <= 'z':
            result.append(chr((ord(ch) - ord('a') + shift) % 26 + ord('a')))
        else:
            result.append(ch)

    return ''.join(result)

decoded = caesar_letters(text)

print(decoded[:2000])

:3 &  %()25( 7+(+21 %/(05-867,&(,$$16$5,  &LAIMINGTOBEPERPLE[EDATTHEDECISIONOFTHE6TATE*OVERNMENTANDCHALL ENGINGTHEVIRUSANDCONSTITUTIONALIT\OFTHE6TATECABINET SDECISIONTOSELECT ANDNOMINATECANDIDATESFORPURSUINGTHECOURSESOFMBBSAND%'6ONTHEBASISO FAPOLIC\WHICHPROVIDESFORMAKINGSELECTIONANDNOMINATIONOFTHOSECANDIDAT ESWHODONOTCOMEWITHINTHE]ONEOFCONSIDERATIONONTHEBASISOFTHEMERITL ISTPREPAREDFOLLOWINGTHE-OINT(NTRANCE([AMINATION HEREINAFTERINSHORTCA LLEDAS?THE-((? BUTAREFROMSUCHDISTRICTS S FROMWHICHNOCANDIDATECOUL DBESELECTEDONTHEBASISOFMERITALONEFORPURSUINGTHESAIDCOURSESTHEPRI NCIPLETHUSBEINGTHATEACHDISTRICTOFTHE6TATEMUSTGETREPRESENTATIONIFA N\OFITSCANDIDATEHASRECEIVEDTHEREQUISITEQUALIF\INGMARKSTHOUGHHEMA\B EFARLOWERTHANOTHERSINTHEMERITLISTTHEPETITIONERSHAVEAPPROACHEDTHIS &OURT  %\THISAPPLICATIONMADEUNDERARTICLE

## quantify the corruption

In [20]:
import pandas as pd
import re

def control_count(text):
    return len(re.findall(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', str(text)))

df["control_chars"] = df["full_text"].apply(control_count)

print("=== CORRUPTION BY COURT ===")
print(
    df.groupby("court_code")["control_chars"]
      .agg(
          documents="count",
          affected=lambda x: (x > 0).sum(),
          avg_control="mean",
          max_control="max"
      )
      .sort_values("affected", ascending=False)
      .to_string()
)

=== CORRUPTION BY COURT ===
            documents  affected  avg_control  max_control
court_code                                               
18_6            12491       379    30.914659         9758
24_17            8973        39     0.004458            2
33_10              47        26     1.595745           15
3_22              397        17    74.256927         5375
1_12              180         4    45.927778         6116
16_20            3307         4     0.001814            2
8_9                52         3    15.384615          382
10_8            11877         1     0.150543         1788
23_23             305         1     0.006557            2
20_7               10         0     0.000000            0
19_16            5661         0     0.000000            0
14_25            2376         0     0.000000            0
17_21            1859         0     0.000000            0
11_24             322         0     0.000000            0
27_1              905         0     0.000000

In [21]:
print("\n=== OVERALL ===")
print("Total documents:", len(df))
print("Affected:", (df["control_chars"] > 0).sum())
print(
    "Affected %:",
    round((df["control_chars"] > 0).mean() * 100, 2)
)


=== OVERALL ===
Total documents: 50000
Affected: 474
Affected %: 0.95


## create the cleaned corpus

In [22]:
import re
import pandas as pd
from pathlib import Path

def control_count(text):
    return len(re.findall(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', str(text)))

df["control_chars"] = df["full_text"].apply(control_count)

# Keep documents with <= 1% control characters.
# Severely corrupted documents are excluded.
df_clean = df[
    (df["control_chars"] / df["text_length"]) <= 0.01
].copy()

print("Original:", len(df))
print("Clean:", len(df_clean))
print("Removed:", len(df) - len(df_clean))

Original: 50000
Clean: 49629
Removed: 371


In [23]:
# normalization
def normalize_text(text):
    text = str(text)

    # Remove control characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df_clean["clean_text"] = df_clean["full_text"].apply(normalize_text)

In [24]:
output = Path("/content/legalrag_corpus/legal_judgments_clean.parquet")

df_clean.to_parquet(output, index=False)

print("Saved:", output)
print("Documents:", len(df_clean))

Saved: /content/legalrag_corpus/legal_judgments_clean.parquet
Documents: 49629


## Create baseline chunks

In [25]:
!pip -q install langchain-text-splitters

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

df = pd.read_parquet(
    "/content/legalrag_corpus/legal_judgments_clean.parquet"
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for _, row in df.iterrows():
    text = row["clean_text"]

    for i, chunk in enumerate(splitter.split_text(text)):
        chunks.append({
            "chunk_id": f"{row['cnr']}_{i}",
            "cnr": row["cnr"],
            "court_code": row["court_code"],
            "decision_date": row["decision_date"],
            "case_type": row["case_type"],
            "title": row["title"],
            "chunk_index": i,
            "text": chunk,
        })

chunks_df = pd.DataFrame(chunks)

print("Documents:", len(df))
print("Chunks:", len(chunks_df))
print(
    "Average chunks/document:",
    round(len(chunks_df) / len(df), 2)
)

Documents: 49629
Chunks: 304364
Average chunks/document: 6.13


In [27]:
# Inspect the result
print(chunks_df.head(3)[
    ["chunk_id", "chunk_index", "text"]
].to_string(index=False))

                       chunk_id  chunk_index                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [28]:
# Inspect the result
chunk_lengths = chunks_df["text"].str.len()

print("\nChunk length statistics:")
print(chunk_lengths.describe())


Chunk length statistics:
count    304364.000000
mean       1017.721560
std         223.576255
min           4.000000
25%         984.000000
50%        1099.000000
75%        1159.000000
max        1200.000000
Name: text, dtype: float64


In [29]:
# Remove extremely small chunks
before = len(chunks_df)

chunks_df = chunks_df[
    chunks_df["text"].str.len() >= 100
].copy()

after = len(chunks_df)

print("Removed:", before - after)
print("Remaining chunks:", after)

Removed: 630
Remaining chunks: 303734


In [30]:
# save the chunks
chunks_df.to_parquet(
    "/content/legalrag_corpus/baseline_chunks.parquet",
    index=False
)

print("Saved baseline chunks.")

Saved baseline chunks.


In [31]:
print(chunks_df["text"].str.len().describe())

count    303734.000000
mean       1019.682067
std         219.617886
min         100.000000
25%         985.000000
50%        1099.000000
75%        1159.000000
max        1200.000000
Name: text, dtype: float64


# Baseline chunking is now locked:
- 49,629 documents
- 303,734 usable chunks
- 1,200-char chunks
- 200-char overlap
- minimum chunk length = 100

# Evaluation Dataset

In [32]:
import random
import pandas as pd

random.seed(42)

# Sample 20 judgments initially for inspection
sample_docs = df.sample(20, random_state=42)

for _, row in sample_docs.iterrows():
    print("=" * 100)
    print("CNR:", row["cnr"])
    print("TITLE:", row["title"])
    print("\nTEXT:")
    print(row["clean_text"][:2500])

CNR: GJHC240161901992_1_1996-10-08
TITLE: CR.A/379/1992 of STATE OF GUJARAT Vs CHHAGANBHAI GOVINDBHAI

TEXT:
IN THE HIGH COURT OF GUJARAT AT AHMEDABAD CRIMINAL APPEAL No 379 of 1992 with CRIMINAL REVISION APPLICATION No 335 of 1992 For Approval and Signature: Hon'ble MR.JUSTICE A.N.DIVECHA ============================================================ 1. Whether Reporters of Local Papers may be allowed to see the judgements? Yes 2. To be referred to the Reporter or not? Yes 3. Whether Their Lordships wish to see the fair copy of the judgement? No 4. Whether this case involves a substantial question of law as to the interpretation of the Constitution of India, 1950 of any Order made thereunder? No 5. Whether it is to be circulated to the Civil Judge? No -------------------------------------------------------------- STATE OF GUJARAT Versus CHHAGANBHAI GOVINDBHAI -------------------------------------------------------------- Appearance: Shri M.A. Bukhari, Additional Public Prosecutor, for t

## build the evaluation set

In [33]:
import pandas as pd

df = pd.read_parquet(
    "/content/legalrag_corpus/legal_judgments_clean.parquet"
)

# Select 100 judgments
eval_docs = df.sample(100, random_state=42).copy()

eval_docs.to_parquet(
    "/content/legalrag_corpus/evaluation_documents.parquet",
    index=False
)

print("Evaluation documents:", len(eval_docs))

Evaluation documents: 100


In [34]:
print(eval_docs[["cnr", "title"]].head(10).to_string(index=False))

                          cnr                                                                                   title
GJHC240161901992_1_1996-10-08                             CR.A/379/1992 of STATE OF GUJARAT Vs CHHAGANBHAI GOVINDBHAI
BRHC010711042021_1_2025-12-02                       CR. MISC./61065/2021 of RAJESH KUMAR RANJAN Vs THE STATE OF BIHAR
GAHC040008562025_1_2025-06-02                    WP(C)/236/2025 of Smti Gichak Daniam Vs The Union of India and 2 Ors
WBCHCA0384292025_1_2025-09-24                       CRM (A)/2947/2025 of BIMAN DALUI AND ANR. Vs STATE OF WEST BENGAL
GAHC010089982008_1_2015-12-15 WP(C)/5215/2008 of ASSAM RAJYA RASTRABHASA PRACHAR SAMITI Vs THE UNION OF INDIA and ORS
BRHC010962002025_1_2025-11-25               CR. MISC./69793/2025 of Rakesh Mahto @ Rakesh Kumar Vs The State of Bihar
GJHC240247911998_1_1998-07-22                    CR.A/121/1998 of MADHUBEN W/O SHANABHAI FULABHAI Vs STATE OF GUJARAT
WBCHCJ0027072023_1_2023-06-14                           

## 10-question pilot

In [35]:
!pip -q install google-genai

In [36]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API key: ")

Enter Gemini API key:  ········


In [37]:
import json
import time
import pandas as pd
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Use the 10 judgments we already selected
pilot_docs = eval_docs.head(10).copy()

results = []

PROMPT_TEMPLATE = """
You are creating a high-quality evaluation benchmark for a legal RAG system.

Using ONLY the Indian court judgment below, create ONE question that:
1. Requires understanding the judgment.
2. Can be answered from the judgment.
3. Is not answerable merely from the case number, date, judge, or party names.
4. Focuses on one of:
   - legal reasoning
   - facts relevant to the decision
   - legal provision/application
   - court's final outcome
5. Does not introduce information not present in the judgment.

Return ONLY valid JSON in exactly this format:

{
  "question": "...",
  "reference_answer": "...",
  "question_type": "fact|reasoning|legal_provision|outcome|multi_hop",
  "supporting_text": "Exact or near-exact passage from the judgment supporting the answer."
}

The supporting_text must be taken from the judgment itself.

JUDGMENT:
"""

for i, (_, row) in enumerate(pilot_docs.iterrows(), start=1):

    print(f"Generating {i}/10...")

    prompt = PROMPT_TEMPLATE + "\n" + row["clean_text"]

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
        )

        text = response.text.strip()

        # Remove accidental markdown fences
        if text.startswith("```"):
            text = text.replace("```json", "").replace("```", "").strip()

        item = json.loads(text)

        item["cnr"] = row["cnr"]

        results.append(item)

        time.sleep(1)

    except Exception as e:
        print(f"Failed for {row['cnr']}: {e}")

print("\nGenerated:", len(results))

Generating 1/10...
Generating 2/10...
Generating 3/10...
Generating 4/10...
Generating 5/10...
Generating 6/10...
Generating 7/10...
Generating 8/10...
Generating 9/10...
Generating 10/10...

Generated: 10


In [38]:
#Inspect the questions
for i, item in enumerate(results, start=1):
    print("=" * 90)
    print(f"{i}. {item['question']}")
    print("TYPE:", item["question_type"])
    print("ANSWER:", item["reference_answer"])
    print("SUPPORT:", item["supporting_text"][:500])

1. On what grounds did the learned Additional Sessions Judge reduce the respondent-accused's sentence, and why did the High Court reject those grounds?
TYPE: reasoning
ANSWER: The Additional Sessions Judge reduced the sentence on two grounds: (1) that the respondent-accused was carrying dairy milk only as a carrier, and (2) that he was a poor person. The High Court rejected the first ground because the accused did not raise it when the Food Inspector tried to collect the sample, failed to disclose the milk owner's name, and did not put this defense in the Food Inspector's cross-examination, having only raised it during his further statement under section 313 of the Code. The High Court rejected the second ground because poverty did not even figure in the accused's statement under section 313 (making it a figment of the appellate judge's imagination) and because poverty cannot be considered an adequate or special reason to reduce a substantive sentence below the minimum prescribed under

In [39]:
#Save the pilot
pilot_df = pd.DataFrame(results)

pilot_df.to_json(
    "/content/legalrag_corpus/eval_pilot_10.json",
    orient="records",
    indent=2
)

print("Saved:", len(pilot_df))

Saved: 10


## connect each question to its gold chunk

In [40]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def normalize(s):
    return re.sub(r"\s+", " ", str(s)).strip()

gold_results = []

for item in results:
    cnr = item["cnr"]

    # Get original clean judgment
    doc = df[df["cnr"] == cnr]

    if len(doc) == 0:
        print("Document not found:", cnr)
        continue

    full_text = normalize(doc.iloc[0]["clean_text"])
    support = normalize(item["supporting_text"])

    # Locate supporting passage
    start = full_text.find(support)

    if start == -1:
        print("\nCould not exactly locate support:", cnr)
        print("Support preview:", support[:200])
        continue

    end = start + len(support)

    # Recreate chunks
    chunks = splitter.split_text(full_text)

    gold_chunk_ids = []

    search_start = 0

    for i, chunk in enumerate(chunks):
        chunk_norm = normalize(chunk)

        chunk_start = full_text.find(chunk_norm, search_start)

        if chunk_start == -1:
            continue

        chunk_end = chunk_start + len(chunk_norm)

        # Does this chunk overlap the supporting passage?
        if chunk_start < end and chunk_end > start:
            gold_chunk_ids.append(f"{cnr}_{i}")

        search_start = chunk_start + 1

    gold_results.append({
        "cnr": cnr,
        "question": item["question"],
        "reference_answer": item["reference_answer"],
        "question_type": item["question_type"],
        "supporting_text": item["supporting_text"],
        "gold_chunk_ids": gold_chunk_ids
    })

    print(
        f"{cnr}: "
        f"{len(gold_chunk_ids)} gold chunk(s)"
    )


Could not exactly locate support: GJHC240161901992_1_1996-10-08
Support preview: The learned Additional Sessions Judge has reduced the sentence merely on the ground that the respondent-accused was carrying dairy milk as a carrier and he was a poor person... So far as the second gr
BRHC010711042021_1_2025-12-02: 2 gold chunk(s)
GAHC040008562025_1_2025-06-02: 1 gold chunk(s)
WBCHCA0384292025_1_2025-09-24: 2 gold chunk(s)

Could not exactly locate support: GAHC010089982008_1_2015-12-15
Support preview: If the petitioners are still aggrieved, they may approach the appropriate departmental authority by filing appropriate application. In the event of such approach being made, the said authority shall p
BRHC010962002025_1_2025-11-25: 2 gold chunk(s)
GJHC240247911998_1_1998-07-22: 1 gold chunk(s)
WBCHCJ0027072023_1_2023-06-14: 1 gold chunk(s)
WBCHCP0013652023_1_2023-11-24: 2 gold chunk(s)
BRHC010443172025_1_2025-12-02: 1 gold chunk(s)


In [41]:
for x in gold_results:
    print("=" * 80)
    print("Q:", x["question"])
    print("Gold chunks:", x["gold_chunk_ids"])

Q: What is the reason for the disposal of the Criminal Miscellaneous petition filed for the cancellation of regular bail?
Gold chunks: ['BRHC010711042021_1_2025-12-02_0', 'BRHC010711042021_1_2025-12-02_1']
Q: What specific relief did the petitioner seek regarding the payment of her balance GST liability, and what statutory provision was invoked in support of this request?
Gold chunks: ['GAHC040008562025_1_2025-06-02_2']
Q: What specific act and weapon formed the basis of the serious allegations against the petitioners, as pointed out by the State from the case diary?
Gold chunks: ['WBCHCA0384292025_1_2025-09-24_0', 'WBCHCA0384292025_1_2025-09-24_1']
Q: What reason did the learned A.P.P. give to oppose the petitioner's claim of parity with other co-accused persons who were granted bail?
Gold chunks: ['BRHC010962002025_1_2025-11-25_1', 'BRHC010962002025_1_2025-11-25_2']
Q: What specific circumstances and items of evidence did the court rely on to establish that the appellant deliberately

## semantic/lexical gold-chunk matching

In [42]:
from difflib import SequenceMatcher

def find_best_chunks(cnr, supporting_text, top_n=3):
    candidates = chunks_df[chunks_df["cnr"] == cnr].copy()

    support = supporting_text.lower()

    def score(text):
        return SequenceMatcher(
            None,
            support,
            text.lower()
        ).ratio()

    candidates["match_score"] = candidates["text"].apply(score)

    return candidates.sort_values(
        "match_score",
        ascending=False
    ).head(top_n)[
        ["chunk_id", "match_score", "text"]
    ]

In [43]:
##Test the failed example:

failed = results[1]

print(find_best_chunks(
    failed["cnr"],
    failed["supporting_text"],
    top_n=3
).to_string(index=False))

                       chunk_id  match_score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [44]:
#the other:

failed = results[6]

print(find_best_chunks(
    failed["cnr"],
    failed["supporting_text"],
    top_n=3
).to_string(index=False))

                       chunk_id  match_score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

## create the 100-question benchmark

### gold-chunk assignment automatic

In [45]:
def assign_gold_chunk(cnr, supporting_text):
    candidates = chunks_df[chunks_df["cnr"] == cnr].copy()

    if candidates.empty:
        return None, 0.0

    support = normalize(supporting_text)

    candidates["score"] = candidates["text"].apply(
        lambda x: SequenceMatcher(
            None,
            support.lower(),
            normalize(x).lower()
        ).ratio()
    )

    best = candidates.sort_values(
        "score",
        ascending=False
    ).iloc[0]

    return best["chunk_id"], best["score"]

In [46]:
validated = []

for item in results:
    chunk_id, score = assign_gold_chunk(
        item["cnr"],
        item["supporting_text"]
    )

    if score >= 0.50:
        item["gold_chunk_ids"] = [chunk_id]
        item["gold_match_score"] = score
        validated.append(item)

print("Validated:", len(validated))

Validated: 6


In [47]:
import json

with open(
    "/content/legalrag_corpus/eval_pilot_validated.json",
    "w"
) as f:
    json.dump(validated, f, indent=2)

print("Saved:", len(validated))

Saved: 6


##  Generate the remaining 92 questions in batches of 10

In [48]:
# Existing successful candidates
existing_cnrs = {x["cnr"] for x in results}

remaining_docs = eval_docs[
    ~eval_docs["cnr"].isin(existing_cnrs)
].copy()

print("Existing:", len(existing_cnrs))
print("Remaining:", len(remaining_docs))

Existing: 10
Remaining: 90


In [49]:
from pydantic import BaseModel
from typing import List, Literal
from google.genai import types
import json
import time


class EvalItem(BaseModel):
    cnr: str
    question: str
    reference_answer: str
    question_type: Literal[
        "fact",
        "reasoning",
        "legal_provision",
        "outcome",
        "multi_hop"
    ]
    supporting_text: str


class EvalBatch(BaseModel):
    items: List[EvalItem]

BATCH_PROMPT = """
You are generating a high-quality benchmark for a legal RAG system.

For EACH judgment below, create exactly ONE evaluation question.

Rules:
- Use ONLY the supplied judgment.
- Question must require understanding, not just metadata lookup.
- Prefer reasoning, legal provision/application, facts relevant to decision,
  or outcome.
- Do not invent information.
- Reference answer must be supported by the judgment.
- supporting_text must be copied or closely extracted from the judgment.
- Return exactly one item for each CNR.

JUDGMENTS:
"""

## generate in batches of 5

In [50]:
remaining_docs = eval_docs[
    ~eval_docs["cnr"].isin({x["cnr"] for x in results})
].copy()

for start in range(0, len(remaining_docs), 5):

    batch = remaining_docs.iloc[start:start + 5]

    prompt_parts = [BATCH_PROMPT]

    for _, row in batch.iterrows():
        prompt_parts.append(
            f"\nCNR: {row['cnr']}\n"
            f"JUDGMENT:\n{row['clean_text']}\n"
        )

    prompt = "\n".join(prompt_parts)

    print(f"Batch {start//5 + 1}: {len(batch)} judgments")

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=EvalBatch,
            temperature=0.1,
        ),
    )

    parsed = EvalBatch.model_validate_json(response.text)

    for item in parsed.items:
        results.append(item.model_dump())

    with open(
        "/content/legalrag_corpus/eval_candidates.json",
        "w"
    ) as f:
        json.dump(results, f, indent=2)

    print("Total candidates:", len(results))

    time.sleep(2)

Batch 1: 5 judgments
Total candidates: 15
Batch 2: 5 judgments
Total candidates: 20
Batch 3: 5 judgments
Total candidates: 25
Batch 4: 5 judgments
Total candidates: 30
Batch 5: 5 judgments
Total candidates: 35
Batch 6: 5 judgments
Total candidates: 40
Batch 7: 5 judgments
Total candidates: 45
Batch 8: 5 judgments
Total candidates: 50
Batch 9: 5 judgments
Total candidates: 55
Batch 10: 5 judgments
Total candidates: 60
Batch 11: 5 judgments
Total candidates: 65
Batch 12: 5 judgments
Total candidates: 70
Batch 13: 5 judgments
Total candidates: 75
Batch 14: 5 judgments
Total candidates: 80
Batch 15: 5 judgments
Total candidates: 85
Batch 16: 5 judgments
Total candidates: 90
Batch 17: 5 judgments
Total candidates: 95
Batch 18: 5 judgments
Total candidates: 100


In [51]:
import json
import pandas as pd

# Load the latest saved progress
with open(
    "/content/legalrag_corpus/eval_candidates.json"
) as f:
    results = json.load(f)

print("Already generated:", len(results))

existing_cnrs = {x["cnr"] for x in results}

remaining_docs = eval_docs[
    ~eval_docs["cnr"].isin(existing_cnrs)
].copy()

print("Remaining:", len(remaining_docs))

Already generated: 100
Remaining: 0


In [56]:
for _, row in remaining_docs.iterrows():

    print("Generating:", row["cnr"])

    prompt = BATCH_PROMPT + (
        f"\nCNR: {row['cnr']}\n"
        f"JUDGMENT:\n{row['clean_text']}\n"
    )

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=EvalBatch,
                temperature=0.1,
            ),
        )

        parsed = EvalBatch.model_validate_json(response.text)

        for item in parsed.items:
            results.append(item.model_dump())

        with open(
            "/content/legalrag_corpus/eval_candidates.json",
            "w"
        ) as f:
            json.dump(results, f, indent=2)

        print("Total:", len(results))

    except Exception as e:
        print("Failed:", e)

# Gold batch generation

In [57]:
import re

def normalize_text(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()


def find_gold_chunks(cnr, supporting_text):
    doc_chunks = chunks_df[chunks_df["cnr"] == cnr].copy()

    support = normalize_text(supporting_text)

    if doc_chunks.empty:
        return []

    # Find the longest meaningful phrase from the support
    words = support.split()

    # Try progressively smaller windows until we find an exact phrase
    matches = []

    for window in [60, 40, 25, 15, 10]:
        if len(words) < window:
            continue

        for i in range(len(words) - window + 1):
            phrase = " ".join(words[i:i + window])

            for _, chunk in doc_chunks.iterrows():
                chunk_text = normalize_text(chunk["text"])

                if phrase in chunk_text:
                    matches.append(chunk["chunk_id"])

            if matches:
                break

        if matches:
            break

    return list(dict.fromkeys(matches))

In [58]:
"""validated = []

for item in candidates:

    gold_chunks = find_gold_chunks(
        item["cnr"],
        item["supporting_text"]
    )

    item["gold_chunk_ids"] = gold_chunks

    if gold_chunks:
        validated.append(item)

print("Candidates:", len(candidates))
print("With evidence:", len(validated))
print("Without evidence:", len(candidates) - len(validated))"""

NameError: name 'candidates' is not defined

In [59]:
import json
import re
import pandas as pd

# Load the generated evaluation candidates
with open("/content/legalrag_corpus/eval_candidates.json", "r") as f:
    candidates = json.load(f)

print("Candidates:", len(candidates))

Candidates: 100


In [60]:
import json

# Deduplicate by CNR and keep only grounded questions
final_eval = []
seen = set()

for item in validated:
    cnr = item["cnr"]

    if cnr not in seen and item["gold_chunk_ids"]:
        final_eval.append(item)
        seen.add(cnr)

print("Final evaluation questions:", len(final_eval))

Final evaluation questions: 0


In [61]:
with open(
    "/content/legalrag_corpus/gold_eval.json",
    "w"
) as f:
    json.dump(final_eval, f, indent=2)

print("Saved gold_eval.json")

Saved gold_eval.json


In [62]:
from collections import Counter

print("Question types:")
print(Counter(x["question_type"] for x in final_eval))

print("\nQuestions with gold evidence:")
print(sum(bool(x["gold_chunk_ids"]) for x in final_eval))

Question types:
Counter()

Questions with gold evidence:
0


# BM25

In [63]:
!pip -q install rank-bm25

## Build BM25

In [64]:
from rank_bm25 import BM25Okapi

corpus = chunks_df["text"].tolist()

tokenized_corpus = [
    text.lower().split()
    for text in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

print("Indexed chunks:", len(corpus))

Indexed chunks: 303734


In [77]:
import json

with open(
    "/content/legalrag_corpus/gold_eval.json",
    "r"
) as f:
    final_eval = json.load(f)

print("Evaluation questions:", len(final_eval))

Evaluation questions: 0


## Retrieval function

In [75]:
def bm25_search(query, k=5):
    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    top_indices = scores.argsort()[-k:][::-1]

    return chunks_df.iloc[top_indices].copy()

In [76]:
test_results = bm25_search(
    final_eval[0]["question"],
    k=5
)

print(
    test_results[
        ["chunk_id", "cnr", "text"]
    ].to_string(index=False)
)

IndexError: list index out of range

## Evaluate Recall@K

In [69]:
import numpy as np

def recall_at_k(item, k):
    retrieved = bm25_search(item["question"], k)
    retrieved_ids = set(retrieved["chunk_id"])
    gold_ids = set(item["gold_chunk_ids"])

    return int(bool(retrieved_ids & gold_ids))

In [70]:
# Retrieve TOP-10 only once for every evaluation question

all_top10 = {}

for i, item in enumerate(final_eval, start=1):
    all_top10[item["cnr"]] = bm25_search(
        item["question"],
        k=10
    )

    if i % 10 == 0:
        print(f"Processed {i}/{len(final_eval)}")

In [71]:
import numpy as np

for k in [1, 3, 5, 10]:

    scores = []

    for item in final_eval:

        retrieved = all_top10[item["cnr"]].head(k)

        retrieved_ids = set(retrieved["chunk_id"])
        gold_ids = set(item["gold_chunk_ids"])

        scores.append(
            int(bool(retrieved_ids & gold_ids))
        )

    print(f"Recall@{k}: {np.mean(scores):.4f}")

Recall@1: nan
Recall@3: nan
Recall@5: nan
Recall@10: nan


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## Freeze this result

In [ ]:
bm25_metrics = {
    "retriever": "BM25",
    "recall@1": 0.20,
    "recall@3": 0.24,
    "recall@5": 0.28,
    "recall@10": 0.33
}

import json

with open(
    "/kaggle/working/bm25_baseline.json",
    "w"
) as f:
    json.dump(bm25_metrics, f, indent=2)

print("BM25 baseline saved.")

# dense retrieval baseline

In [ ]:
!pip -q install sentence-transformers faiss-gpu

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Load the embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

print("Model loaded")

## Generate embeddings in batches



In [ ]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())]

print("Using:", devices)

pool = model.start_multi_process_pool(
    target_devices=devices
)

embeddings = model.encode_multi_process(
    texts,
    pool=pool,
    batch_size=64,
    normalize_embeddings=True,
)

model.stop_multi_process_pool(pool)

print("Embedding shape:", embeddings.shape)